# StudyAbroadGPT 512-token Re-Generation

This notebook is a **drop-in replacement** for the 256-token evaluation in `evaluate_studyabroadgpt_lora.ipynb`, with one change: `max_new_tokens=256` -> `max_new_tokens=512`. Everything else is held fixed so the only thing that varies between the two runs is the generation cap.

## Why we are re-running

The 256-token run produced a 96% truncation rate (48/50 base, 48/50 LoRA). That floor-censors any automatic metric that depends on the response actually finishing (caveat-phrase coverage, completeness, hallucination-risk language at the tail). Re-running at 512 tokens removes the cap and lets us report a defensible head-to-head.

## What this notebook writes

All outputs go to `/kaggle/working/eval_512/` and are also zipped for download:

- `base_model_outputs_512.csv`
- `lora_model_outputs_512.csv`
- `downstream_raw_model_outputs_512.csv`
- `downstream_generation_metadata_512.csv`
- `evaluation_config_512.json`
- `automatic_sanity_metrics_512.csv`
- `studyabroadgpt_512_outputs.zip` (final bundle)

## What stays the same as the 256-token run

- Base model: `mistralai/Mistral-7B-Instruct-v0.3`
- LoRA model: `millat/StudyAbroadGPT-7B-LoRa-Kaggle` (subfolder `merged`)
- Dataset: `millat/StudyAbroadGPT-Dataset`, split `test`
- Random seed: 42, sample size: 50
- Generation: `do_sample=False`, `temperature=0.0`, `top_p=1.0`
- Quantization: 4-bit NF4 via `bitsandbytes`
- Prompt template: Mistral chat template (fallback: `<s>[INST] ... [/INST]`)

## 1. Install dependencies

In [ ]:
!pip install -q transformers accelerate datasets bitsandbytes torch pandas tqdm sentencepiece peft

## 2. Configuration

In [ ]:
import os
import gc
import json
import time
import random
from datetime import datetime, timezone
from pathlib import Path

import torch
import pandas as pd
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

OUTPUT_DIR = Path('/kaggle/working/eval_512')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BASE_MODEL_ID = 'mistralai/Mistral-7B-Instruct-v0.3'
LORA_MODEL_ID = 'millat/StudyAbroadGPT-7B-LoRa-Kaggle'
LORA_MODEL_SUBFOLDER = 'merged'
DATASET_ID = 'millat/StudyAbroadGPT-Dataset'

RANDOM_SEED = 42
N_SAMPLES = 50

# *** The only change from the original eval: max_new_tokens 256 -> 512 ***
GENERATION_CONFIG = {
    'max_new_tokens': 512,
    'do_sample': False,
    'temperature': 0.0,
    'top_p': 1.0,
}

random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed_all(RANDOM_SEED)

print('Output dir :', OUTPUT_DIR)
print('Base model :', BASE_MODEL_ID)
print('LoRA model :', LORA_MODEL_ID, '(subfolder:', LORA_MODEL_SUBFOLDER, ')')
print('Dataset    :', DATASET_ID, '(split: test)')
print('Sample size:', N_SAMPLES)
print('Generation :', GENERATION_CONFIG)
print('CUDA       :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU        :', torch.cuda.get_device_name(0))

## 3. Load 50 held-out prompts (same seed as the 256-token run)

In [ ]:
def first_user_prompt(conversation):
    for turn in conversation:
        if turn.get('from') == 'human':
            return str(turn.get('value', '')).strip()
    return ''

raw_test = load_dataset(DATASET_ID, split='test')
rng = random.Random(RANDOM_SEED)
indices = list(range(len(raw_test)))
rng.shuffle(indices)
selected_indices = indices[:N_SAMPLES]

samples = []
for sample_id, idx in enumerate(selected_indices, start=1):
    item = raw_test[idx]
    prompt = first_user_prompt(item.get('conversations', []))
    if not prompt:
        continue
    samples.append({
        'sample_id': sample_id,
        'dataset_index': idx,
        'prompt': prompt,
    })

prompts_df = pd.DataFrame(samples)
prompts_df.to_csv(OUTPUT_DIR / 'evaluation_prompts_512.csv', index=False)
print('Usable prompts:', len(prompts_df))
display(prompts_df.head(3))

## 4. Model loaders and generation helpers

In [ ]:
def clear_gpu_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def build_quantization_config():
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )


def load_model_and_tokenizer(model_id: str, subfolder: str | None = None, base_model_id: str | None = None):
    """Load a model. Three code paths:

    1. (Base model) Pass `subfolder=None` and the public HF repo id. Loads
       with 4-bit NF4 quantization. This is the path for `BASE_MODEL_ID`.

    2. (Merged LoRA model) Pass `subfolder='merged'` and the LoRA repo id.
       The `merged/` subfolder contains the merged FP16 weights, not a
       PEFT adapter. We download the merged folder via `snapshot_download`
       and load it directly with the same 4-bit NF4 config. This is the
       path for `LORA_MODEL_ID`.

    3. (PEFT adapter) Pass a different `subfolder` (or none) and the LoRA
       repo id. If the subfolder actually contains `adapter_config.json`,
       load the base in 4-bit and apply the PEFT adapter on top.

    This avoids the failure mode where the merged folder is mistakenly
    treated as a PEFT adapter and 404s on `adapter_config.json`.
    """
    quant_config = build_quantization_config()
    common_model_kwargs = {
        'device_map': 'auto',
        'torch_dtype': torch.float16,
        'low_cpu_mem_usage': True,
        'quantization_config': quant_config,
        'trust_remote_code': True,
    }
    common_tokenizer_kwargs = {
        'use_fast': True,
        'trust_remote_code': True,
    }

    # Path 2: merged-weights subfolder. Snapshot-download then load.
    if subfolder == 'merged':
        from huggingface_hub import snapshot_download
        print(f'  snapshot_download {model_id} subfolder={subfolder} ...')
        local_dir = snapshot_download(
            model_id,
            allow_patterns=[f'{subfolder}/*', '*.json', '*.txt', '*.model'],
        )
        merged_path = str(Path(local_dir) / subfolder)
        print(f'  loading merged weights from {merged_path}')

        tokenizer = AutoTokenizer.from_pretrained(merged_path, use_fast=True)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        tokenizer.padding_side = 'left'

        model = AutoModelForCausalLM.from_pretrained(merged_path, **common_model_kwargs)
        model.eval()
        return model, tokenizer

    # Path 1: plain HF repo. Try loading with the subfolder as-is.
    if subfolder:
        common_model_kwargs['subfolder'] = subfolder
        common_tokenizer_kwargs['subfolder'] = subfolder
    tokenizer = AutoTokenizer.from_pretrained(model_id, **common_tokenizer_kwargs)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = 'left'

    try:
        model = AutoModelForCausalLM.from_pretrained(model_id, **common_model_kwargs)
        model.eval()
        return model, tokenizer
    except OSError as exc:
        msg = str(exc)
        if 'does not appear to have a file named' not in msg:
            raise
        # Path 3: PEFT adapter. Load base in 4-bit, then apply adapter.
        from huggingface_hub import list_repo_files
        try:
            repo_files = list_repo_files(model_id)
        except Exception as hub_exc:
            raise RuntimeError(
                f'Failed to load {model_id} directly and could not list repo files: {hub_exc}'
            ) from hub_exc
        adapter_relpath = f'{subfolder}/adapter_config.json' if subfolder else 'adapter_config.json'
        if adapter_relpath not in repo_files:
            raise OSError(
                f'{model_id} subfolder={subfolder!r} has neither merged weights nor an adapter_config.json. '
                f'Inspect https://huggingface.co/{model_id}/tree/main to see what is shipped.'
            ) from exc

        base_model_id = base_model_id or BASE_MODEL_ID
        print(f'  loading base {base_model_id} (4-bit NF4) and applying PEFT adapter from {model_id} subfolder={subfolder}')
        base_model = AutoModelForCausalLM.from_pretrained(base_model_id, **common_model_kwargs)
        from peft import PeftModel
        peft_kwargs = {}
        if subfolder:
            peft_kwargs['subfolder'] = subfolder
        model = PeftModel.from_pretrained(base_model, model_id, **peft_kwargs)
        model.eval()
        return model, tokenizer

## 5. Generate base model responses (512 tokens)

In [ ]:
prompts = prompts_df['prompt'].tolist()

base_results = generate_responses_for_model(
    BASE_MODEL_ID,
    prompts,
    GENERATION_CONFIG,
    label='base model (512)',
)

base_outputs_df = pd.DataFrame([
    {
        'sample_id': row['sample_id'],
        'dataset_index': row['dataset_index'],
        'prompt': row['prompt'],
        'response': r['response'],
        'generation_time_sec': r['generation_time_sec'],
        'response_char_length': r['response_char_length'],
        'response_token_count': r['response_token_count'],
        'was_truncated': r['was_truncated'],
    }
    for row, r in zip(prompts_df.to_dict('records'), base_results)
])
base_outputs_df.to_csv(OUTPUT_DIR / 'base_model_outputs_512.csv', index=False)
print('Base responses saved:', len(base_outputs_df))

## 6. Generate LoRA/merged model responses (512 tokens)

In [ ]:
lora_results = generate_responses_for_model(
    LORA_MODEL_ID,
    prompts,
    GENERATION_CONFIG,
    label='LoRA/merged model (512)',
    subfolder=LORA_MODEL_SUBFOLDER,
    base_model_id=BASE_MODEL_ID,
)

lora_outputs_df = pd.DataFrame([
    {
        'sample_id': row['sample_id'],
        'dataset_index': row['dataset_index'],
        'prompt': row['prompt'],
        'response': r['response'],
        'generation_time_sec': r['generation_time_sec'],
        'response_char_length': r['response_char_length'],
        'response_token_count': r['response_token_count'],
        'was_truncated': r['was_truncated'],
    }
    for row, r in zip(prompts_df.to_dict('records'), lora_results)
])
lora_outputs_df.to_csv(OUTPUT_DIR / 'lora_model_outputs_512.csv', index=False)
print('LoRA responses saved:', len(lora_outputs_df))

## 7. Build merged outputs, metadata, and config

In [ ]:
records = []
for b_row, l_row in zip(base_outputs_df.to_dict('records'), lora_outputs_df.to_dict('records')):
    records.append({
        'sample_id': b_row['sample_id'],
        'dataset_index': b_row['dataset_index'],
        'prompt': b_row['prompt'],
        'base_model_response': b_row['response'],
        'lora_model_response': l_row['response'],
        'base_generation_time_sec': b_row['generation_time_sec'],
        'lora_generation_time_sec': l_row['generation_time_sec'],
        'base_response_char_length': b_row['response_char_length'],
        'lora_response_char_length': l_row['response_char_length'],
        'base_response_token_count': b_row['response_token_count'],
        'lora_response_token_count': l_row['response_token_count'],
        'base_was_truncated': b_row['was_truncated'],
        'lora_was_truncated': l_row['was_truncated'],
    })
results_df = pd.DataFrame(records)
results_df.to_csv(OUTPUT_DIR / 'downstream_raw_model_outputs_512.csv', index=False)

metadata_df = results_df[[
    'sample_id', 'dataset_index',
    'base_generation_time_sec', 'lora_generation_time_sec',
    'base_response_char_length', 'lora_response_char_length',
    'base_response_token_count', 'lora_response_token_count',
    'base_was_truncated', 'lora_was_truncated',
]]
metadata_df.to_csv(OUTPUT_DIR / 'downstream_generation_metadata_512.csv', index=False)

config = {
    'base_model_id': BASE_MODEL_ID,
    'lora_model_id': LORA_MODEL_ID,
    'lora_model_subfolder': LORA_MODEL_SUBFOLDER,
    'dataset_id': DATASET_ID,
    'split': 'test',
    'sample_size': int(N_SAMPLES),
    'usable_prompt_count': int(len(results_df)),
    'random_seed': int(RANDOM_SEED),
    'generation_config': GENERATION_CONFIG,
    'timestamp_utc': datetime.now(timezone.utc).isoformat(),
    'raw_outputs_path': str(OUTPUT_DIR / 'downstream_raw_model_outputs_512.csv'),
    'metadata_path': str(OUTPUT_DIR / 'downstream_generation_metadata_512.csv'),
    'paired_with_256_run': {
        '256_metadata': '/content/drive/MyDrive/LoRA_Paper/outputs/downstream_generation_metadata.csv',
        'note': 'Same 50 prompts, same seed, same models, only max_new_tokens differs (256 -> 512).',
    },
}
with open(OUTPUT_DIR / 'evaluation_config_512.json', 'w') as f:
    json.dump(config, f, indent=2)
print('Saved raw outputs, metadata, and config.')

## 8. Sanity metrics (truncation rate is the headline number here)

In [ ]:
sanity = {
    'base_avg_response_char_length': results_df['base_response_char_length'].mean(),
    'lora_avg_response_char_length': results_df['lora_response_char_length'].mean(),
    'base_avg_response_token_count': results_df['base_response_token_count'].mean(),
    'lora_avg_response_token_count': results_df['lora_response_token_count'].mean(),
    'base_avg_generation_time_sec': results_df['base_generation_time_sec'].mean(),
    'lora_avg_generation_time_sec': results_df['lora_generation_time_sec'].mean(),
    'base_truncation_rate': results_df['base_was_truncated'].mean(),
    'lora_truncation_rate': results_df['lora_was_truncated'].mean(),
}
sanity_df = pd.DataFrame([sanity])
sanity_df.to_csv(OUTPUT_DIR / 'automatic_sanity_metrics_512.csv', index=False)

print('=' * 60)
print('Headline: truncation rate at max_new_tokens=512')
print('=' * 60)
print(f"  base : {sanity['base_truncation_rate']*100:.1f}% (was 96.0% at 256)")
print(f"  lora : {sanity['lora_truncation_rate']*100:.1f}% (was 96.0% at 256)")
print()
print('Avg response length (chars):')
print(f"  base : {sanity['base_avg_response_char_length']:.1f}  (was 1151.88 at 256)")
print(f"  lora : {sanity['lora_avg_response_char_length']:.1f}  (was 1178.74 at 256)")
print()
print('Avg response length (tokens):')
print(f"  base : {sanity['base_avg_response_token_count']:.1f}")
print(f"  lora : {sanity['lora_avg_response_token_count']:.1f}")
print()
print('Avg generation time (sec):')
print(f"  base : {sanity['base_avg_generation_time_sec']:.2f}")
print(f"  lora : {sanity['lora_avg_generation_time_sec']:.2f}")

## 9. Bundle outputs for download

In [ ]:
import shutil
zip_path = '/kaggle/working/studyabroadgpt_512_outputs'
shutil.make_archive(zip_path, 'zip', OUTPUT_DIR)
print('Zipped:', zip_path + '.zip')
print('\nFiles inside:')
for p in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {p.name}  ({p.stat().st_size:,} bytes)")